# 08 Target Resume Human Modification

## Purpose

This notebook supports human review and modification of a generated target resume content artifact.

It operates after target resume assembly and before rendering. The goal is to provide a safe, auditable interface for final content decisions without editing raw JSON by hand and without mixing content changes with layout or PDF rendering.

## Inputs

Primary input:

```text
target_resume_content_vN.json
```

Optional supporting inputs for addition recommendations:

```text
canonical_master_resume.json
candidate_evidence.json
selected_evidence.json
target_archetype.json
resume_positioning.json
```

## Outputs

This notebook may produce:

```text
target_resume_content_vN_manual_review.txt
target_resume_content_vM.json
target_resume_manual_edit_log_vM.json
target_resume_revision_log_vM.json
```

where `vM` is the next content version.

## Supported modifications

### 1. Manual wording edits

The notebook exports selected resume text into a plain text review file. The human edits wording in that file, then imports the edits back into the JSON structure.

Use this for wording polish only:

```text
Tighten a sentence.
Replace awkward phrasing.
Correct tone.
Remove overclaiming.
Improve clarity.
```

Manual wording edits should not add or delete bullets.

### 2. Selected content additions

The notebook can display ranked addition recommendations and apply human-selected priority numbers.

Use this after reviewing a rendered resume and deciding that additional evidence would improve length or signal.

## Not currently supported

Structured bullet removal is not yet supported. For now, removal should be handled by regenerating with different selections, manually editing JSON, or adding removal support in a later version.

## Future stub: LLM-assisted wording suggestions

A later version may support targeted wording assistance for one text block at a time, such as:

```text
Make block T0042 five words shorter.
Reword block T0031 to be clearer.
Reduce overclaiming in block T0027.
```

The LLM should only suggest replacement text. The human must approve before anything is written back to JSON.

## Versioning rule

Each modification pass should produce a new content version.

```text
target_resume_content_v1.json -> initial generated content
target_resume_content_v2.json -> manual wording edits
target_resume_content_v3.json -> selected additions
target_resume_content_v4.json -> final wording polish
```

When practical, each version should represent one kind of change.


In [1]:
%run ./init_notebook.py

Repo root: /Users/douglasdaly/GitHub/Generative-AI
Added src to sys.path: /Users/douglasdaly/GitHub/Generative-AI/src
Resume builder notebooks: /Users/douglasdaly/GitHub/Generative-AI/notebooks/resume-builder
Artifacts: /Users/douglasdaly/GitHub/Generative-AI/notebooks/resume-builder/artifacts


In [2]:
from genai_demos.resume_builder.content_review import (
    add_additional_content,
    display_addition_recommendations,
    display_target_resume_text,
    export_manual_review_text,
    import_manual_review_text,
    load_target_content,
    next_content_version,
)
from genai_demos.resume_builder.content_recommendations import (
    get_target_profile,
    rank_content_additions
)
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

load_dotenv()


True

## 8A Controls

Set the base content version, the next version to write, and which workflow sections should run.


In [3]:
BASE_CONTENT_VERSION = "v4"
NEXT_CONTENT_VERSION = "v5"

RUN_DISPLAY_CONTENT = True

RUN_EXPORT_MANUAL_REVIEW = False
RUN_IMPORT_MANUAL_REVIEW = True

RUN_GENERATE_CHANGE_RECOMMENDATIONS = False
RUN_LOAD_CHANGE_RECOMMENDATIONS = False
RUN_DISPLAY_CHANGE_RECOMMENDATIONS = False
RUN_APPLY_SELECTED_CHANGES = False

RUN_LLM_REWORDING_STUB = False
APPROVED_CHANGE_PRIORITIES = []


MODEL = ChatOpenAI(
    model="gpt-4.1",
    temperature=0,
)

HUMAN_REVISION_REQUEST = """
The rendered resume has room for additional content.
Recommend the highest-value content additions for the target role.
Choose between adding professional experience bullets and adding selected projects
based on which option provides the greatest additional signal.
"""

## 8B Load target resume content

In [4]:
target_resume_content, resolved_base_version = load_target_content(
    ARTIFACT_DIR,
    version=BASE_CONTENT_VERSION,
)

resolved_next_version = NEXT_CONTENT_VERSION or next_content_version(resolved_base_version)

print(f"Loaded content version: {resolved_base_version}")
print(f"Next content version:   {resolved_next_version}")


Loaded content version: v4
Next content version:   v5


## 8C View current content

This is a readable plain-text display for review and sanity checking. It does not modify artifacts.


In [5]:
if RUN_DISPLAY_CONTENT:
    display_target_resume_text(target_resume_content)


Douglas Gordon Daly


Senior technical leader with deep expertise architecting enterprise AI/ML platforms, governed GenAI and tool-calling systems, and operational intelligence solutions. Built governed analytics assistants, monitoring frameworks, and reusable data/AI platforms with strong emphasis on controls, observability, and auditable execution. Experienced across cloud-native AI infrastructure, large-scale data platforms, executive alignment, and technical mentorship. Known for translating ambiguous business needs into scalable, reliable analytical systems that improve decision-making and operational performance.


- AI Platform Architecture - enterprise AI platforms, governed analytics assistants, operational intelligence systems, reusable AI/analytics frameworks
- GenAI & Tool-Calling Systems - LLM applications, structured outputs, tool orchestration, RAG/GraphRAG systems, prompt engineering, evaluation, traceability
- Governance & Operational Controls - responsible AI adoption

## 8D Export manual wording review file

This creates a plain text file with stable block IDs and JSON paths.

Edit only the text after `TEXT:` in each block. Do not edit `@@TEXT`, `PATH_JSON`, `TYPE`, `SOURCE`, or `@@END` lines.


In [6]:
manual_review_path = ARTIFACT_DIR / f"target_resume_content_{resolved_base_version}_manual_review.txt"

if RUN_EXPORT_MANUAL_REVIEW:
    review_blocks = export_manual_review_text(
        target_resume_content=target_resume_content,
        review_text_path=manual_review_path,
    )

    print(f"Exported {len(review_blocks)} editable text blocks.")
    print(f"Manual review file: {manual_review_path}")
    print()
    print("Edit only the text after TEXT:. Do not add/delete blocks or change PATH_JSON.")


## 8E Import manual wording edits

After editing the review text file, run this cell to write a new target resume content JSON version and an edit log.


In [7]:
if RUN_IMPORT_MANUAL_REVIEW:
    updated_resume_content, manual_edit_log = import_manual_review_text(
        base_resume_content=target_resume_content,
        review_text_path=manual_review_path,
    )

    updated_content_path = ARTIFACT_DIR / f"target_resume_content_{resolved_next_version}.json"
    manual_edit_log_path = ARTIFACT_DIR / f"target_resume_manual_edit_log_{resolved_next_version}.json"

    save_json(updated_resume_content, updated_content_path)
    save_json(manual_edit_log, manual_edit_log_path)

    print(f"Saved updated content: {updated_content_path}")
    print(f"Saved manual edit log: {manual_edit_log_path}")
    print()
    print(f"Changed blocks: {manual_edit_log['changed_count']}")
    print(f"Unchanged blocks: {manual_edit_log['unchanged_count']}")

    manual_edit_log


Saved updated content: /Users/douglasdaly/GitHub/Generative-AI/notebooks/resume-builder/artifacts/target_resume_content_v5.json
Saved manual edit log: /Users/douglasdaly/GitHub/Generative-AI/notebooks/resume-builder/artifacts/target_resume_manual_edit_log_v5.json

Changed blocks: 3
Unchanged blocks: 40


## 8F Generate addition recommendations

In [8]:
recommendations_version = resolved_base_version
recommendations_path = ARTIFACT_DIR / f"target_resume_addition_recommendations_{recommendations_version}.json"

candidate_evidence_path = ARTIFACT_DIR / "candidate_evidence.json"
selected_evidence_path = ARTIFACT_DIR / "selected_evidence.json"
resume_positioning_path = ARTIFACT_DIR / "resume_positioning.json"
canonical_resume_path = ARTIFACT_DIR / "canonical_master_resume.json"

if RUN_GENERATE_CHANGE_RECOMMENDATIONS:
    candidate_evidence = load_json(candidate_evidence_path)
    selected_evidence = load_json(selected_evidence_path)
    resume_positioning = load_json(resume_positioning_path)
    canonical_resume = load_json(canonical_resume_path)

    print(f"Loaded candidate evidence: {candidate_evidence_path.name}")
    print(f"Loaded selected evidence:  {selected_evidence_path.name}")
    print(f"Loaded positioning:        {resume_positioning_path.name}")

    target_profile = get_target_profile(resume_positioning)

    change_recommendations = rank_content_additions(
        current_resume_content=target_resume_content,
        canonical_resume=canonical_resume,
        candidate_evidence=candidate_evidence,
        selected_evidence=selected_evidence,
        target_profile=target_profile,
        human_revision_request=HUMAN_REVISION_REQUEST,
        model=MODEL,
    )

    change_recommendations["_metadata"] = {
        "recommendation_type": "addition_recommendations",
        "base_content_version": resolved_base_version,
        "supported_actions": [
            "add_role_bullet",
            "add_selected_project",
        ],
        "notes": (
            "Generated for the specified base content version. "
            "Do not apply to a different base version without regenerating."
        ),
    }

    save_json(change_recommendations, recommendations_path)

    print(f"Saved change recommendations: {recommendations_path}")

## 8G Display addition recommendations

This section assumes a prior ranking step has saved `target_resume_addition_recommendations_vN.json`.

A later version can move the ranking prompt itself into this notebook. For now, this section gives a cleaner interface for reviewing and applying recommendation output.


In [9]:
if RUN_DISPLAY_CHANGE_RECOMMENDATIONS:
    change_recommendations = load_json(recommendations_path)
    display_addition_recommendations(change_recommendations)


## 8G Apply selected additions

The LLM recommends additions. The human chooses which priority numbers to apply.

Set `APPROVED_ADDITION_PRIORITIES` to a list of priority numbers from the displayed recommendations.


In [10]:
if RUN_APPLY_SELECTED_CHANGES:
    updated_resume_content, revision_log = add_additional_content(
        base_resume_content=target_resume_content,
        addition_recommendations=change_recommendations,
        approved_priorities=APPROVED_CHANGE_PRIORITIES,
        base_content_version=resolved_base_version,
        next_content_version=NEXT_CONTENT_VERSION,
    )

    updated_content_path = ARTIFACT_DIR / f"target_resume_content_{NEXT_CONTENT_VERSION}.json"
    revision_log_path = ARTIFACT_DIR / f"target_resume_revision_log_{NEXT_CONTENT_VERSION}.json"

    save_json(updated_resume_content, updated_content_path)
    save_json(revision_log, revision_log_path)

    print(f"Saved updated content: {updated_content_path}")
    print(f"Saved revision log:    {revision_log_path}")

    revision_log

## 8H Future: LLM-assisted wording suggestions

Stub only. Keep the first implementation deterministic and auditable.


In [11]:
if RUN_LLM_REWORDING_STUB:
    raise NotImplementedError(
        "LLM-assisted rewording is planned for a later version. "
        "For now, use the manual review text file for wording edits."
    )
